<a href="https://colab.research.google.com/github/thamiherzing/BillRetrieval/blob/main/Bill_Retrival.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

%pip install requests pandas beautifulsoup4 openpyxl


In [3]:
from pathlib import Path

excel_files = list(Path("/content").glob("*.xlsx"))

print("Excel files uploaded:")
for file in excel_files:
    print(file.name)


Excel files uploaded:
Bills 1.xlsx


In [6]:
import pandas as pd
from openpyxl import load_workbook

file_path = "/content/Bills 1.xlsx"

# Read the entire master workbook
master_bills = pd.read_excel(file_path)
master_bills = master_bills.rename(
    columns={"Unnamed: 1": "Bill_Number"}
)

# Select only the new 90 entries, using your existing research IDs
new_ids = [f"USA_{n:04d}" for n in range(37, 127)]

bills = master_bills.loc[
    master_bills["Bill_ID"].isin(new_ids)
].copy()

if len(bills) != 90 or set(bills["Bill_ID"]) != set(new_ids):
    raise ValueError(
        "Expected USA_0037–USA_0126 exactly once each. "
        "Check that the updated master workbook is uploaded."
    )

# Read the Congress.gov hyperlinks stored in column B.
# pandas reads cell text but does not retain these hyperlinks.
workbook = load_workbook(file_path)
sheet = workbook.worksheets[0]

links = {
    row[0].value: row[1].hyperlink.target
    for row in sheet.iter_rows(min_row=2)
    if row[1].hyperlink
}

bills["Congress_URL"] = bills["Bill_ID"].map(links)
workbook.close()

# Extract Congress, bill type and number from each official URL
parts = bills["Congress_URL"].str.extract(
    r"/bill/(\d+)th-congress/(house|senate)-bill/(\d+)"
)
parts.columns = ["Congress", "Chamber", "Number"]

invalid = parts.isna().any(axis=1)

if invalid.any():
    display(
        bills.loc[invalid, ["Bill_ID", "Bill_Number", "Congress_URL"]]
    )
    raise ValueError("These entries need valid Congress.gov hyperlinks.")

bills["Congress"] = parts["Congress"].astype(int)
bills["Bill_Type"] = parts["Chamber"].map(
    {"house": "hr", "senate": "s"}
)
bills["Bill_Number_Numeric"] = parts["Number"].astype(int)

bills["Congress_Bill_ID"] = (
    bills["Congress"].astype(str)
    + "-"
    + bills["Bill_Type"]
    + "-"
    + bills["Bill_Number_Numeric"].astype(str)
)

display(bills[["Bill_ID", "Bill_Number", "Congress_Bill_ID"]])

print("Selected entries:", len(bills))
print("Unique congressional bills:", bills["Congress_Bill_ID"].nunique())
print(
    "Duplicate congressional IDs:",
    bills["Congress_Bill_ID"].duplicated().sum()
)

,Bill_ID,Bill_Number,Congress_Bill_ID
36,USA_0037,H.R. 2,115-hr-2
37,USA_0038,H.R. 6,115-hr-6
38,USA_0039,H.R. 4760,115-hr-4760
39,USA_0040,H.R. 6136,115-hr-6136
40,USA_0041,H.R. 5515,115-hr-5515
...,...,...,...
121,USA_0122,H.R. 3746,118-hr-3746
122,USA_0123,H.R. 2670,118-hr-2670
123,USA_0124,H.R. 5860,118-hr-5860
124,USA_0125,H.R. 6363,118-hr-6363


Selected entries: 90
Unique congressional bills: 89
Duplicate congressional IDs: 1


In [7]:
from getpass import getpass

congress_api_key = getpass("Insert your key here").strip()

if not congress_api_key:
    raise ValueError("No key entered. Run this cell again.")

print("Key stored for this Colab session. Ready to test the connection.")

Insert your key here··········
Key stored for this Colab session. Ready to test the connection.


In [8]:
import requests

# Use the first bill from your selected 90 entries
first_bill = bills.iloc[0]

url = (
    f"https://api.congress.gov/v3/bill/"
    f"{int(first_bill['Congress'])}/"
    f"{first_bill['Bill_Type']}/"
    f"{int(first_bill['Bill_Number_Numeric'])}"
)

response = requests.get(
    url,
    params={"api_key": congress_api_key, "format": "json"},
    timeout=30
)

print("Research ID:", first_bill["Bill_ID"])
print("Congressional ID:", first_bill["Congress_Bill_ID"])
print("Connection status:", response.status_code)

if response.status_code == 200:
    bill_info = response.json()["bill"]
    print("Bill:", bill_info["type"], bill_info["number"])
    print("Title:", bill_info["title"])
else:
    print("The request did not succeed. Send me the status number.")

Research ID: USA_0037
Congressional ID: 115-hr-2
Connection status: 200
Bill: HR 2
Title: Agriculture Improvement Act of 2018


In [9]:
actions = []
offset = 0

actions_url = (
    f"https://api.congress.gov/v3/bill/"
    f"{int(first_bill['Congress'])}/"
    f"{first_bill['Bill_Type']}/"
    f"{int(first_bill['Bill_Number_Numeric'])}/actions"
)

while True:
    response = requests.get(
        actions_url,
        params={
            "api_key": congress_api_key,
            "format": "json",
            "limit": 250,
            "offset": offset
        },
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Request failed with status {response.status_code}"
        )

    data = response.json()
    batch = data.get("actions", [])
    actions.extend(batch)

    if not data.get("pagination", {}).get("next"):
        break

    if not batch:
        raise RuntimeError("An empty page was returned unexpectedly.")

    offset += len(batch)

actions_table = pd.DataFrame(
    [
        {
            "Date": action.get("actionDate"),
            "Action": action.get("text"),
            "Record references": action.get("links", [])
        }
        for action in actions
    ],
    columns=["Date", "Action", "Record references"]
)

print("Research ID:", first_bill["Bill_ID"])
print("Congressional ID:", first_bill["Congress_Bill_ID"])

display(actions_table.sort_values("Date"))
print("Total actions retrieved:", len(actions))

Research ID: USA_0037
Congressional ID: 115-hr-2


,Date,Action,Record references
190,2018-04-12,Introduced in House,[]
187,2018-04-12,Referred to the House Committee on Agriculture.,[]
189,2018-04-12,Introduced in House,[]
188,2018-04-12,Sponsor introductory remarks on measure. (CR H...,[]
186,2018-04-18,Committee Consideration and Mark-up Session Held.,[]
...,...,...,...
4,2018-12-19,Presented to President.,[]
3,2018-12-20,Signed by President.,[]
2,2018-12-20,Signed by President.,[]
1,2018-12-20,Became Public Law No: 115-334.,[]


Total actions retrieved: 191


In [10]:
dates = pd.to_datetime(actions_table["Date"], errors="coerce")
texts = actions_table["Action"].fillna("")

within_period = dates.between("2018-01-01", "2023-12-31")

house_debate = (
    texts.str.contains(r"\bDEBATE\b", case=False)
    & texts.str.contains(
        r"\bHouse\b|\bCommittee of the Whole\b",
        case=False
    )
)

house_record_reference = texts.str.contains(
    r"\bCR\s+H\d",
    case=False
)

candidate_actions = actions_table.loc[
    within_period & (house_debate | house_record_reference)
].sort_values("Date")

# Print full text so references are not cut off
for _, row in candidate_actions.iterrows():
    print(row["Date"])
    print(row["Action"])
    print("-" * 80)

print("Matching actions:", len(candidate_actions))
print("Candidate dates:", candidate_actions["Date"].nunique())

2018-04-12
Sponsor introductory remarks on measure. (CR H3196-3204)
--------------------------------------------------------------------------------
2018-05-16
DEBATE - Pursuant to the provisions of H. Res. 891, the Committee of the Whole proceeded with 10 minutes of debate on the Thompson (PA) amendment No. 9.
--------------------------------------------------------------------------------
2018-05-16
DEBATE - Pursuant to the provisions of H. Res. 891, the Committee of the Whole proceeded with 10 minutes of debate on the Latta amendment No. 8.
--------------------------------------------------------------------------------
2018-05-16
DEBATE - Pursuant to the provisions of H. Res. 891, the Committee of the Whole proceeded with 10 minutes of debate on the Jones amendment No. 7.
--------------------------------------------------------------------------------
2018-05-16
DEBATE - Pursuant to the provisions of H. Res. 891, the Committee of the Whole proceeded with 10 minutes of debate on the

In [11]:
# Exclude actions that do not establish floor consideration.
exclude_from_floor_search = candidate_actions["Action"].str.contains(
    r"Sponsor introductory remarks"
    r"|Conference report.*filed"
    r"|Rules Committee Resolution",
    case=False,
    na=False
)

floor_candidate_actions = candidate_actions.loc[
    ~exclude_from_floor_search
].copy()

candidate_dates = sorted(
    floor_candidate_actions["Date"].dropna().unique().tolist()
)

print("Bill:", first_bill["Congress_Bill_ID"])
print("Candidate House dates:", len(candidate_dates))

for date in candidate_dates:
    print(date)

Bill: 115-hr-2
Candidate House dates: 6
2018-05-16
2018-05-17
2018-05-18
2018-06-21
2018-07-18
2018-12-12
